In [ ]:
import pandas as pd
import json
import scanpy as sc
import os


from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("curation.log"),
        logging.StreamHandler(),  # keep console output too
    ],
    force=True,
)
# show all columns
pd.set_option('display.max_columns', None)

# Download data

In [ ]:
from concurrent.futures import ThreadPoolExecutor

file_suffices = [
    'IFNB',
    'IFNG', 
    'INS', 
    'TGFB', 
    'TNFA'
    ]
with ThreadPoolExecutor(max_workers=min(5, len(file_suffices))) as executor:
    futures = [
        executor.submit(
            download_file,
            url=f"https://zenodo.org/records/14518762/files/Seurat_object_{suffix}_Perturb_seq.rds?download=1",
            dest_path=f'../non_curated/jiang_2025_{suffix}.rds',
            unarchive=True,
        )
        for suffix in file_suffices
    ]

    for future in futures:
        future.result()

### Convert to h5ad

The data is available only as .rds files native for R. We will convert them to h5ad files using the `scCustomize::as.anndata` function in R.

Run the following script `data_exploration/Perturbseq/supplementary/jiang_2025/convert_to_h5ad.R <input_rds_file> <output_h5ad_file>` for each .rds file to convert it to .h5ad format. 

The converted files should be saved in the `data_exploration/Perturbseq/curation_notebooks/jiang_2025/non_curated/h5ad` directory.

Note that this operation requires R, `scCustomize` and `reticulate` packages to be installed.

The conversion requires up to 160 GB of RAM, depending on the size of the .rds file. 

### Concatenate h5ad files

After converting all .rds files to .h5ad format, we will concatenate them into a single h5ad file for easier downstream analysis.

Uncomment the code in the cell below and run it to concatenate all h5ad files in the `data_exploration/Perturbseq/curation_notebooks/jiang_2025/non_curated/h5ad` directory into a single h5ad file named `data_exploration/Perturbseq/curation_notebooks/jiang_2025/curated/jiang_2025.h5ad`.

In [ ]:
# Collect AnnData objects from the non_curated/h5ad folder
anndatas = []

for f in os.listdir('../non_curated/h5ad'):
    # only process files that start with the study prefix
    if f.startswith('jiang_2025'):
        # read the h5ad file into an AnnData object
        anndata = sc.read_h5ad(f'../non_curated/h5ad/{f}')
        print(f"Loaded {f}")
        print(anndata)
        
        # keep only the desired obs columns to standardize across files
        anndata.obs = anndata.obs[['sample', 'cell_type', 'sample_ID', 'Batch_info', 'guide', 'gene']]
        
        # remove the layers attribute if present to avoid compatibility/size issues
        if anndata.layers:
            del anndata.layers
            print('Removed layers slot')
        
        print("Subset the columns")
        anndatas.append(anndata)
        
# Concatenate all collected AnnData objects along observations (cells).
# - axis='obs' means stacking cells (rows) from each object
# - join='inner' keeps only variables/columns (genes) common to all objects
adata_combined = sc.concat(
    anndatas, 
    axis='obs', 
    join="inner"
)

# Write the combined AnnData to disk as a single h5ad file
adata_combined.write_h5ad("../non_curated/h5ad/jiang_2025.h5ad")


# Initialise the dataset object

In [ ]:
noncurated_path = '../non_curated/h5ad/jiang_2025.h5ad'
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    noncurated_path=noncurated_path
)

cur_data.load_data()

In [ ]:
cur_data.adata.obs

In [ ]:
cur_data.adata.var

### Filter out cells with unknown treatment

In [ ]:
print(f"Number of cells with unknown treatment: {cur_data.adata.obs.query('sample == \"unknown_unknown\"').shape[0]}")

print("Filtering out cells with unknown treatment...")
cur_data.adata = cur_data.adata[cur_data.adata.obs['sample'] != 'unknown_unknown'].copy()
print(f"Number of cells after filtering: {cur_data.adata.shape[0]}")

### Add cell barcodes

In [ ]:
cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.astype(str) + '_' + cur_data.adata.obs['sample'].astype(str)

print(cur_data.adata.obs[['cell_barcode']].head())
print("Number of duplicated cell barcodes:", cur_data.adata.obs['cell_barcode'].duplicated().sum())

### Add `treatment` column derived from `sample` column

In [ ]:
cur_data.adata.obs['treatment'] = cur_data.adata.obs['sample'].str.split('_').str[1]

### Add guide RNA information

Note that NT - non-targeting control guides are missing. The authors have been contacted to clarify whether these were included in the original data and if so, to provide the guide sequences for these guides.

In [ ]:
# download the guide RNA spreadsheet
download_file(
    url="https://static-content.springer.com/esm/art%3A10.1038%2Fs41556-025-01622-z/MediaObjects/41556_2025_1622_MOESM3_ESM.xlsx",
    dest_path="../supplementary/jiang_2025/jiang_2025_guide_info.xlsx"
)

# read in the guide RNA spreadsheet
# guides for the K562 essential day 6 library are in "TabB_K562_day6_library"
guide_info_dict = pd.read_excel("../supplementary/jiang_2025/jiang_2025_guide_info.xlsx", 
                              header=2,
                              sheet_name=['ST1a_IFNG_gRNAs', 'ST1b_IFNB_gRNAs', 'ST1c_TGFB_gRNAs', 'ST1d_TNFA_gRNAs', 'ST1e_INS_gRNAs'])#, sheet_name="TabC_RPE1_day7_library")
# clean up the keys to keep only the treatment name (IFNG, IFNB, TGFB, TNFA, INS)
guide_info_dict = {k.split('_')[1]: v for k,v in guide_info_dict.items()}
# concatenate the guide info dataframes into a single dataframe with a new column for treatment
guide_info_df = (
    pd.concat(guide_info_dict)
    .reset_index(drop=True)
    .rename(columns={'gRNA Name': 'guide', 'gRNA Sequence': 'guide_sequence'})
    [['guide', 'guide_sequence']]
    .drop_duplicates()
)

# read non-targeting guides - these are missing from the main guide info sheets, but were provided at the request by the authors
non_targeting_guides_df = pd.read_table("../supplementary/jiang_2025/jiang_2025_nt_guides.txt", header=None, names=['guide_sequence'])
# reshape
non_targeting_guides_df = pd.concat([non_targeting_guides_df[non_targeting_guides_df['guide_sequence'].str.startswith('>NTg')].reset_index(drop=True), 
           non_targeting_guides_df[~non_targeting_guides_df['guide_sequence'].str.startswith('>NTg')].reset_index(drop=True)],
          axis=1,
          ignore_index=True).rename(columns={0: 'guide', 1: 'guide_sequence'})

non_targeting_guides_df['guide'] = non_targeting_guides_df['guide'].str.replace('>', '').str.strip()
# remove the common prefix from the guide sequences
non_targeting_guides_df['guide_sequence'] = non_targeting_guides_df['guide_sequence'].str.replace('GTGGAAAGGACGAAACACCG','')
# keep first 20 bases of the guide sequence to match the format in the main guide info sheet
non_targeting_guides_df['guide_sequence'] = non_targeting_guides_df['guide_sequence'].str[:20]

# concatenate the non-targeting guides with the main guide info dataframe
guide_info_df = pd.concat([guide_info_df, non_targeting_guides_df], ignore_index=True)

guide_info_df

In [ ]:
# merge the guide info with the obs dataframe to add guide sequences
cur_data.adata.obs = cur_data.adata.obs.merge(guide_info_df, on='guide', how='left')

### Rename relevant metadata columns

`Sample_ID` is 16 separate libraries (from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8609800) -> `technical_replicate`

`Batch_info` is two separate sequencing events for each condition -> `biological_replicate`

`cell_type` is acually cell line -> `cell_line`

`guide` -> `perturbation_name`

In [ ]:
cur_data.adata.obs = cur_data.adata.obs.rename(columns={'sample_ID': 'technical_replicate', 
                                                        'Batch_info': 'biological_replicate',
                                                        'cell_type': 'cell_line',
                                                        'guide': 'perturbation_name'})

### Standardise perturbation targets

In [ ]:
# replace non targeting guides with "NTg" in the perturbation name to "control_nontargeting"
cur_data.adata.obs['gene'] = cur_data.adata.obs['gene'].replace('NT', 'control_nontargeting')

In [ ]:
cur_data.standardize_genes(
    slot='obs',
    input_column='gene',
    input_column_type='gene_symbol',
    multiple_entries=False
)

### Add `perturbed_target_number` column

In [ ]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

### Encode chromosomes as integers

In [ ]:
cur_data.chromosome_encoding()

In [ ]:
cur_data.show_obs(['perturbation_name', 'perturbed_target_chromosome_encoding'])

### Curate cell line information

In [ ]:
cur_data.standardize_ontology(
    input_column='cell_line',
    column_type='term_name',
    ontology_type='cell_line',
    overwrite=True
)

In [ ]:
# manually map HAP1 and BXPC3
cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line_label'] = cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line'].replace({
    'HAP1': 'HAP-1',
    'BXPC3': 'BxPC-3 cell'
})
cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line_id'] = cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line'].replace({
    'HAP1': 'EFO:0007598',
    'BXPC3': 'CLO:0002065'
})

### Curate cell type information

In [ ]:
cur_data.adata.obs['cell_type'] = cur_data.adata.obs['cell_line'].map({
    'HAP1': 'myeloid cell',
    'K562': 'lymphoblast',
    'A549': 'pulmonary alveolar type 2 cell',
    'HT29': 'enterocyte',
    'BXPC3': None,
    'MCF7': 'luminal epithelial cell of mammary gland'
})

In [ ]:
cur_data.standardize_ontology(
    input_column='cell_type',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

### Curate tissue information


In [ ]:
cur_data.adata.obs['tissue'] = cur_data.adata.obs['cell_line'].map({
    'HAP1': 'bone marrow',
    'K562': 'blood',
    'A549': 'lung',
    'HT29': 'colon',
    'BXPC3': 'pancreas',
    'MCF7': 'breast'
})

In [ ]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

### Curate disease information

In [ ]:
cur_data.adata.obs['disease'] = cur_data.adata.obs['cell_line'].map({
    'HAP1': 'leukemia, myeloid, accelerated-phase',
    'K562': 'chronic myelogenous leukemia, BCR-ABL1 positive',
    'A549': 'lung adenocarcinoma',
    'HT29': 'colon adenocarcinoma',
    'BXPC3': 'pancreatic adenocarcinoma',
    'MCF7': 'invasive breast carcinoma'
})

In [ ]:
cur_data.standardize_ontology(
    input_column='disease',
    column_type='term_name',
    ontology_type='disease',
    overwrite=True
)

### Curate sex information

In [ ]:
cur_data.adata.obs['sex_label'] = cur_data.adata.obs['cell_line'].map({
    'HAP1': 'male',
    'K562': 'female',
    'A549': 'male',
    'HT29': 'female',
    'BXPC3': 'female',
    'MCF7': 'female'
})

### Curate developmental stage information

In [ ]:
cur_data.adata.obs['developmental_stage_label'] = cur_data.adata.obs['cell_line'].map({
    'HAP1': 'adult',
    'K562': 'adult',
    'A549': 'adult',
    'HT29': 'adult',
    'BXPC3': 'senior adult',
    'MCF7': 'senior adult'
})

### Curate treatment information

In [ ]:
cur_data.adata.obs['treatment_label'] = cur_data.adata.obs['treatment'].map({
    'TGFB1': 'TGFB1',
    'INS': 'insulin hormone',
    'TNFA': 'TNFA',
    'IFNB': 'IFNB',
    'IFNG': 'IFNG'
})

cur_data.adata.obs['treatment_id'] = cur_data.adata.obs['treatment'].map({
    'TGFB1': 'PR:P01137',
    'INS': 'PR:000050339',
    'TNFA': 'PR:P01375',
    'IFNB': 'PR:000008924',
    'IFNG': 'PR:P01579'
})

In [ ]:
cur_data.adata.obs

### Add metadata

In [ ]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        #----- dataset -----#
        "dataset_id": cur_data.dataset_id,
        #----- sample -----#
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        #----- perturbation type -----#
        "perturbation_type_label": "CRISPRi",
        "perturbation_type_id": None,
        #----- data modality -----#
        "data_modality": "Perturb-seq", # different from "method_name_label"; more general term - choice of CRISPR, MAVE and Perturb-seq
        #----- significance -----#
        "significant": None,
        "significance_criteria": None,
        #----- score interpretation -----#
        "score_interpretation": None,
        #----- treatment -----#
        # "treatment_label": None,
        # "treatment_id": None,
        #----- replicate -----#
        # "technical_replicate": None,
        # "biological_replicate": None,
        #----- model system -----#
        "model_system_label": "cell_line",
        "model_system_id": None,
        #----- tissue -----#
        # "tissue": "blood",
        #----- cell line -----#
        # "cell_line_label": "K 562 cell",
        # "cell_line_id": None,
        #----- cell type -----#
        # "cell_type_label": "lymphoblast",
        # "cell_type_id": None,
        #----- disease -----#
        # "disease_label": "chronic myelogenous leukemia, BCR-ABL1 positive",
        # "disease_id": None,
        #----- timepoint -----#
        "timepoint": "P13DT0H0M0S",
        #----- species -----#
        "species": "Homo sapiens",
        #----- sex -----#
        # "sex_label": "female",
        "sex_id": None,
        #----- developmental stage -----#
        # "developmental_stage_label": "adult",
        "developmental_stage_id": None,
        #----- study metadata -----#
        "study_title": "Systematic reconstruction of molecular pathway signatures using scalable single-cell perturbation screens",
        "study_uri": "https://doi.org/10.1038/s41556-025-01622-z",
        "study_year": 2025,
        #----- authors -----#
        "first_author": "Longda Jiang",
        "last_author": "Rahul Satija",
        #----- experiment metadata -----#
        "experiment_title": "CRISPRi Perturb-seq of IFNb, IFNg, TNFa, TGFb and insulin pathway regulators in IFNb/IFNg/TNFa/TGFb/insulin-stimulated A549, MCF-7, HT-29, HAP1, K562 and BXPC3 cells.",
        "experiment_summary": """
        The goal of the study was to build a database of molecular response signatures and examine their changes across various cellular contexts. A pooled CRISPRi Perturb-seq screen was performed to characterize the molecular responses to perturbations of known IFNb, IFNg, TNFa, TGFb and insulin pathway regulators. To this end, A549, MCF-7, HT-29, HAP1, K562 and BXPC3 cells were transfected with Doench pathway-sepcific sub-libraries and stimulated with IFNb/IFNg/TNFa/TGFb/insulin for 24h. At this point, the cells were harvested, fixed and frozen for batch processing and sequencing using Ultima Genomics UG100 platform.
        """,
        #----- number of perturbed targets/samples -----#
        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],
        #----- library generation type -----#
        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",
        #----- library generation method -----#
        "library_generation_method_id": None,
        "library_generation_method_label": "dCas9-KRAB-MeCP2",
        #----- enzyme and library delivery method -----#
        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",
        #----- enzyme and library integration state -----#
        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",
        #----- enzyme and library expression control -----#
        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "constitutive transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",
        #----- library name and URI and manufacturer -----#
        "library_name": "Human CRISPR Inhibition Pooled Library (Dolcetto)",
        "library_uri": "https://www.addgene.org/pooled-library/broadgpp-human-crispri-dolcetto/",
        "library_manufacturer": "Doench lab",
        #----- library format -----#
        "library_format_id": None,
        "library_format_label": "pooled",
        #----- library scope -----#
        "library_scope_id": None,
        "library_scope_label": "focused",
        #----- library perturbation type -----#
        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "inhibition",
        #----- library additional metadata -----#
        "library_lentiviral_generation": "3",
        "library_grnas_per_target": "3",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()), # for CRISPR/Perturb-seq
        "library_total_variants": None, # for MAVE
        #----- readout dimensionality -----#
        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",
        #---- readout type -----#
        "readout_type_id": None,
        "readout_type_label": "transcriptomic",
        #----- readout technology -----#
        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",
        #----- method -----#
        "method_name_id": None,
        "method_name_label": "Perturb-seq", # different from "data_modality"; more specific term - specific name of the technique
        "method_uri": None,
        #----- sequencing library kit -----#
        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "Parse Biosciences Evercode Whole Transcriptome Mega v1 kit",
        #----- sequencing platform -----#
        "sequencing_platform_id": None,
        "sequencing_platform_label": "Ultima Genomics UG100",
        #----- sequencing strategy -----#
        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",
        #----- software used for counts-----#
        "software_counts_id": None,
        "software_counts_label": "CellRanger",
        #----- software used for analysis -----#
        "software_analysis_id": None,
        "software_analysis_label": "Seurat",
        #----- reference genome -----#
        "reference_genome_id": None,
        "reference_genome_label": "GRCh38",
        #----- license -----#
        "license_label": "free to use license",
        "license_id": "SWO:1000061",
        #----- external datasets -----#
        "associated_datasets": json.dumps([
            {
                "dataset_accession": "GSE281048",
                "dataset_uri": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE281048",
                "dataset_description": "Raw counts; matrix.mtx, features.tsv, barcodes.tsv",
                "dataset_file_name": "GSE281048_Seurat_object_*_Perturb_seq.rds.gz",
            },
            {
                "dataset_accession": "Seurat_object_*_Perturb_seq.rds",
                "dataset_uri": "https://doi.org/10.5281/zenodo.14518762",
                "dataset_description": "Seurat objects of the Perturb-seq data in .rds format",
                "dataset_file_name": "Seurat_object_*_Perturb_seq.rds",
            }
        ])
    }
)

### Match schema column order

In [ ]:
cur_data.match_schema_columns(slot='obs')

### Validate obs metadata

In [ ]:
cur_data.validate_data(slot='obs', verbose=True)

In [ ]:
cur_data.show_obs(['perturbation_name', 'perturbed_target_symbol', 'perturbed_target_ensg', 'perturbed_target_coord'])

# VAR slot curation

### Standardise genes

In [ ]:
cur_data.adata.var

In [ ]:
cur_data.adata.var['gene_name'] = cur_data.adata.var.index

In [ ]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_name",
    input_column_type="gene_symbol",
    remove_version=True,
    multiple_entries=False
)

### Validate var metadata

In [ ]:
cur_data.validate_data(slot='var')

# Save the dataset

In [ ]:
cur_data.save_curated_data_h5ad()

In [ ]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

# Upload to BigQuery

In [ ]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/jiang_2025_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

# Upload to GC Storage

In [ ]:
!gcloud storage cp ../curated/h5ad/jiang_2025_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/